# Sujet : Analyse financière des dépôts de comptes annuels (NBB CBSO)

Vous avez accès aux dépôts de comptes annuels JSON scrapés depuis l'API NBB
CBSO (`nbb_cbso_api_scraper.py`), stockés sur HDFS sous :

```
<path hdfs>
```


## Objectif du sujet

1. **Extraire** les indicateurs financiers clés de chaque dépôt CSV (un
   dépôt = une entreprise, une année)
2. **Stocker** ces indicateurs dans une collection MongoDB (upsert, pas de
   doublons si vous relancez votre script)
3. **Enrichir** avec le secteur (NACE, `kbo_activity`) et la région
   (`kbo_address`) de chaque entreprise
4. **Analyser** et produire 3 comparaisons : santé financière générale,
   comparaison par secteur, comparaison par région

## les codes de rubrique dont vous aurez besoin (vous pouvez directement les fournir a votre IA pour un simple mapping des codes, pas besoin de creer un fichier dedier)

| Code       | Signification                                          |
|------------|---------------------------------------------------------|
| `70`       | Chiffre d'affaires (absent en schéma abrégé/micro)      |
| `9900`     | Marge brute d'exploitation (proxy si `70` absent)       |
| `60`       | Achats de marchandises                                   |
| `62`       | Rémunérations, charges sociales et pensions              |
| `9901`     | Résultat d'exploitation (~ EBIT)                         |
| `65` / `75`| Charges / produits financiers                            |
| `9904` / `9905` | Résultat net de l'exercice                          |
| `67/77`    | Impôts sur le résultat (charge comptable, pas la trésorerie payée) |
| `10/15`    | Capitaux propres                                         |
| `20/58`    | Total actif                                               |
| `54/58` / `50/53` | Trésorerie (valeurs disponibles / placements)      |
| `170/4` / `43` | Dettes financières (long terme / court terme) |
| `9087`     | Effectif moyen en équivalents temps plein (ETP)          |


**Attention** : un code absent d'un dépôt veut dire "non publié", PAS
"zéro" !!! 

ne remplacez pas silencieusement par 0 sauf quand ça a un sens
métier !!! 

pas de dette financière déclarée = 0 dette financière, ce qui
est différent de "chiffre d'affaires non publié" qui doit rester `None`/`null`  !!!


## 1. Extraire les indicateurs de chaque dépôt

Écrivez une fonction qui, à partir d'un CSV de dépôt, retourne un
dictionnaire des indicateurs utiles pour la suite : chiffre d'affaires,
marge brute (ou son proxy), résultat d'exploitation, résultat net,
capitaux propres, trésorerie, dettes financières, effectif ETP, coût du
personnel, impôts sur le résultat.

Parcourez ensuite tous les dépôts d'une entreprise (toutes les années
disponibles) et assemblez un historique par entreprise.

Créez une collection (par exemple `nbb_financials`) où chaque document
représente **une entreprise pour une année donnée** : `enterprise_number`,
`year`, et tous les indicateurs extraits.


In [1]:
from __future__ import annotations

import csv
import json
import os
import re

from hdfs import InsecureClient

HDFS_URL = os.environ.get("HDFS_URL", "http://localhost:9870")
hdfs_client = InsecureClient(HDFS_URL)
RAW_ROOT = "/data/raw"  # même racine que dans le notebook de scraping (NBB_S)

# ============================================================================
# ⚠️ HYPOTHÈSES À VÉRIFIER SUR UN VRAI FICHIER avant de faire confiance à quoi
# que ce soit ci-dessous : je n'ai pas pu obtenir d'exemple réel de CSV de
# dépôt CBSO pour confirmer les noms de colonnes exacts ni le séparateur.
# `inspect_deposit_csv(path)`, tout en bas de cette cellule, sert précisément
# à vérifier ça en un coup d'oeil sur un premier fichier réel.
# ============================================================================

CODE_COLUMN_CANDIDATES = ["Code", "CODE", "Rubrique", "Concept"]
VALUE_COLUMN_CANDIDATES = ["Waarde", "Valeur", "Value", "Periode", "Bedrag", "Montant"]
CSV_DELIMITER = ";"  # convention belge courante ; à corriger si inspect_deposit_csv dit le contraire


def detect_columns(fieldnames: list[str]) -> tuple[str, str]:
    code_col = next((c for c in CODE_COLUMN_CANDIDATES if c in fieldnames), None)
    value_col = next((c for c in VALUE_COLUMN_CANDIDATES if c in fieldnames), None)
    if not code_col or not value_col:
        raise ValueError(
            f"Colonnes non reconnues dans ce CSV : {fieldnames}. "
            "Ajoutez le bon nom dans CODE_COLUMN_CANDIDATES / VALUE_COLUMN_CANDIDATES."
        )
    return code_col, value_col


def parse_number(raw: str):
    """Nombre au format européen probable (virgule décimale, point ou espace
    pour les milliers), avec parenthèses = négatif (convention comptable)."""
    raw = raw.strip()
    if not raw:
        return None
    negative = raw.startswith("(") and raw.endswith(")")
    if negative:
        raw = raw[1:-1]
    raw = raw.replace(" ", "").replace("\xa0", "")
    if "," in raw and "." in raw:
        raw = raw.replace(".", "").replace(",", ".")
    elif "," in raw:
        raw = raw.replace(",", ".")
    try:
        value = float(raw)
    except ValueError:
        return None
    return -value if negative else value


def parse_deposit_rows(csv_reader: csv.DictReader) -> dict[str, float]:
    """{code_rubrique: valeur}. Une ligne absente ou à valeur vide = code
    absent du dict -> "non publié", jamais silencieusement remplacé par 0."""
    code_col, value_col = detect_columns(csv_reader.fieldnames or [])
    values_by_code: dict[str, float] = {}
    for row in csv_reader:
        code = (row.get(code_col) or "").strip()
        raw_value = (row.get(value_col) or "").strip()
        if not code or not raw_value:
            continue
        parsed = parse_number(raw_value)
        if parsed is not None:
            values_by_code[code] = parsed
    return values_by_code


def read_deposit_values(hdfs_path: str) -> dict[str, float]:
    with hdfs_client.read(hdfs_path, encoding="utf-8-sig") as reader:
        return parse_deposit_rows(csv.DictReader(reader, delimiter=CSV_DELIMITER))


# --- Table des codes de rubrique -> nom d'indicateur ---
#
# Les codes séparés par "/" DANS UNE MÊME entrée (ex. "10/15", "67/77") sont
# des codes composés à part entière du plan comptable belge (une seule ligne
# du bilan), à chercher tels quels. Quand DEUX candidats sont listés (ex.
# trésorerie, dettes financières), c'est une alternative selon le schéma
# (complet vs abrégé) : on essaie le premier, sinon le second.
INDICATOR_CODES: dict[str, list[str]] = {
    "chiffre_affaires": ["70"],
    "marge_brute_exploitation": ["9900"],
    "achats_marchandises": ["60"],
    "couts_personnel": ["62"],
    "resultat_exploitation": ["9901"],
    "charges_financieres": ["65"],
    "produits_financiers": ["75"],
    "impots_resultat": ["67/77"],
    "capitaux_propres": ["10/15"],
    "total_actif": ["20/58"],
    "tresorerie": ["54/58", "50/53"],
    "dettes_financieres": ["170/4", "43"],
    "effectif_etp": ["9087"],
}

# Seule exception à la règle "absent = None" : le sujet précise explicitement
# que l'absence de dette financière déclarée signifie 0 dette (contrairement
# au chiffre d'affaires, où l'absence signifie juste "non publié").
ZERO_IF_ABSENT = {"dettes_financieres"}


def lookup_code(values_by_code: dict[str, float], candidate_codes: list[str]):
    for candidate in candidate_codes:
        if candidate in values_by_code:
            return values_by_code[candidate]
    return None


def compute_net_result(values_by_code: dict[str, float]):
    """9904 = bénéfice, 9905 = perte (grandeur positive dans le plan comptable
    belge) -> résultat net = 9904 si présent, sinon -9905.
    ⚠️ Convention de signe non confirmée sur un vrai dépôt dans cet environnement
    -- à vérifier avec inspect_deposit_csv avant de faire confiance aux graphes."""
    profit = values_by_code.get("9904")
    loss = values_by_code.get("9905")
    if profit is not None:
        return profit
    if loss is not None:
        return -abs(loss)
    return None


def extract_indicators(values_by_code: dict[str, float]) -> dict:
    indicators = {name: lookup_code(values_by_code, codes) for name, codes in INDICATOR_CODES.items()}
    for name in ZERO_IF_ABSENT:
        if indicators[name] is None:
            indicators[name] = 0.0
    indicators["resultat_net"] = compute_net_result(values_by_code)
    return indicators


def inspect_deposit_csv(hdfs_path: str, n_rows: int = 10) -> None:
    """À exécuter en PREMIER sur un vrai fichier avant de lancer l'extraction
    en masse : affiche les colonnes détectées + les premières lignes brutes,
    pour confirmer (ou corriger) CODE_COLUMN_CANDIDATES / VALUE_COLUMN_CANDIDATES
    / CSV_DELIMITER ci-dessus."""
    with hdfs_client.read(hdfs_path, encoding="utf-8-sig") as reader:
        csv_reader = csv.DictReader(reader, delimiter=CSV_DELIMITER)
        print("Colonnes détectées :", csv_reader.fieldnames)
        for i, row in enumerate(csv_reader):
            print(row)
            if i >= n_rows - 1:
                break


# Exemple d'usage, à décommenter sur un vrai chemin avant de lancer la suite :
# inspect_deposit_csv("/data/raw/0693810613/cbso/csvs/2023.csv")


/Users/chancybayedi-mayombo/Downloads/projet final partie I /.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
from pymongo import MongoClient, UpdateOne

mongo_client = MongoClient(os.environ.get("MONGO_URI", "mongodb://localhost:27017"))
db = mongo_client[os.environ.get("MONGO_DB", "kbo")]

db.nbb_financials.create_index([("enterprise_number", 1), ("year", 1)], unique=True)


def list_scraped_enterprises() -> list[str]:
    try:
        return hdfs_client.list(RAW_ROOT)
    except Exception:
        return []


def list_deposit_years(enterprise_number: str) -> list[str]:
    csvs_dir = f"{RAW_ROOT}/{enterprise_number}/cbso/csvs"
    try:
        filenames = hdfs_client.list(csvs_dir)
    except Exception:
        return []
    return [name[:-4] for name in filenames if name.endswith(".csv")]


def build_financial_documents() -> list[dict]:
    """Parcourt tous les dépôts déjà scrapés sur HDFS (toutes entreprises,
    toutes années) et construit un document par (entreprise, année)."""
    documents = []
    for enterprise_number in list_scraped_enterprises():
        for year in list_deposit_years(enterprise_number):
            hdfs_path = f"{RAW_ROOT}/{enterprise_number}/cbso/csvs/{year}.csv"
            try:
                values_by_code = read_deposit_values(hdfs_path)
            except Exception as exc:
                print(f"{enterprise_number}/{year} : erreur de lecture ({exc}), ignoré")
                continue
            documents.append({
                "enterprise_number": enterprise_number,
                "year": int(year),
                **extract_indicators(values_by_code),
            })
    return documents


def store_financials(documents: list[dict]) -> None:
    """Upsert sur (enterprise_number, year) : relancer ce script ne crée jamais
    de doublon, ça met juste à jour les documents déjà présents."""
    if not documents:
        print("Aucun document à stocker.")
        return
    operations = [
        UpdateOne(
            {"enterprise_number": doc["enterprise_number"], "year": doc["year"]},
            {"$set": doc},
            upsert=True,
        )
        for doc in documents
    ]
    result = db.nbb_financials.bulk_write(operations, ordered=False)
    print(f"nbb_financials : {result.upserted_count} inséré(s), {result.modified_count} mis à jour")


financial_documents = build_financial_documents()
store_financials(financial_documents)
print(f"{len(financial_documents)} document(s) (entreprise, année) construits au total")


Aucun document à stocker.
0 document(s) (entreprise, année) construits au total


## 3. Enrichir avec secteur (NACE) et région

Chaque document `nbb_financials` doit pouvoir être rattaché à :
- un/des **secteur/s** : via `kbo_activity` (`Classification == "MAIN"` uniquement)
- une **région** : via `kbo_address` (le code postal ou la ville permet de
  déduire Flandre / Wallonie / Bruxelles)


In [3]:
# On enrichit via `entreprise_silver` (déjà construite dans les notebooks
# précédents de ce projet) plutôt que de retraduire kbo_activity/kbo_address à
# la main : le secteur (activities.main) et l'adresse du siège y sont déjà
# résolus et lisibles. Si votre entreprise_silver n'est pas à jour, la
# variante "directe" (via kbo_activity/kbo_address bruts + traduction code.csv,
# comme dans le TD silver) reste équivalente si vous préférez repartir de zéro.

# Découpage des codes postaux belges par province (mapping standard) :
POSTAL_RANGES_TO_PROVINCE = [
    (1000, 1299, "Bruxelles-Capitale"),
    (1300, 1499, "Brabant wallon"),
    (1500, 1999, "Brabant flamand"),
    (2000, 2999, "Anvers"),
    (3000, 3499, "Brabant flamand"),
    (3500, 3999, "Limbourg"),
    (4000, 4999, "Liège"),
    (5000, 5999, "Namur"),
    (6000, 6599, "Hainaut"),
    (6600, 6999, "Luxembourg"),
    (7000, 7999, "Hainaut"),
    (8000, 8999, "Flandre occidentale"),
    (9000, 9999, "Flandre orientale"),
]

PROVINCE_TO_REGION = {
    "Bruxelles-Capitale": "Bruxelles",
    "Brabant wallon": "Wallonie", "Liège": "Wallonie", "Namur": "Wallonie",
    "Hainaut": "Wallonie", "Luxembourg": "Wallonie",
    "Brabant flamand": "Flandre", "Anvers": "Flandre", "Limbourg": "Flandre",
    "Flandre occidentale": "Flandre", "Flandre orientale": "Flandre",
}


def zipcode_to_province(zipcode: str):
    if not zipcode or not zipcode.isdigit():
        return None
    code = int(zipcode)
    for low, high, province in POSTAL_RANGES_TO_PROVINCE:
        if low <= code <= high:
            return province
    return None


def zipcode_to_region(zipcode: str):
    province = zipcode_to_province(zipcode)
    return PROVINCE_TO_REGION.get(province)


def enrich_financials() -> None:
    pipeline = [
        {"$lookup": {"from": "entreprise_silver", "localField": "enterprise_number", "foreignField": "_id", "as": "silver"}},
        {"$unwind": {"path": "$silver", "preserveNullAndEmptyArrays": True}},
    ]
    updated = 0
    for doc in db.nbb_financials.aggregate(pipeline):
        silver = doc.get("silver") or {}

        main_activities = (silver.get("activities") or {}).get("main", [])
        sectors = sorted({activity["description"] for activity in main_activities})

        siege = (silver.get("addresses") or {}).get("Siège", {})
        province = zipcode_to_province(siege.get("zipcode", ""))
        region = PROVINCE_TO_REGION.get(province)

        db.nbb_financials.update_one(
            {"_id": doc["_id"]},
            {"$set": {"sectors": sectors, "province": province, "region": region}},
        )
        updated += 1

    print(f"{updated} document(s) nbb_financials enrichi(s) (secteur + province + région)")


enrich_financials()


0 document(s) nbb_financials enrichi(s) (secteur + province + région)


## 4. Les 3 analyses demandées

Le support est **entièrement libre**, tableau, graphe, texte commenté,
ce qui vous semble le plus lisible pour chaque cas. Ce qui compte, c'est
que la comparaison soit claire et SIMPLE.

1. **Comparaison générale de la santé financière** : sur l'ensemble des
   entreprises (2021 à aujourd'hui), qu'observez-vous sur la solvabilité
   (capitaux propres / total actif), la trésorerie, l'évolution du
   résultat net dans le temps ? Y a-t-il une tendance globale (ex: reprise
   post-2021, ralentissement plus récent, taux d'interet entre 2022-2024 et les dettes) ?
2. **Comparaison par secteur** : regroupez par code NACE et comparez au moins 3 indicateurs (
   résultat d'exploitation moyen, coût du personnel relatif, solvabilité). (attention, on a pas toujours les indicateurs necessaire pour faire les comparaisons !)
   Quels secteurs s'en sortent le mieux/moins bien ?
3. **Comparaison par région** : même question que **secteur**, mais en regroupant par
   région/province au lieu du secteur.

Pour chacune des 3 analyses, une phrase de conclusion suffit a la fin de vos graphs/tableau (pas un rapport IA SVP)


In [11]:
pip install pandas matplotlib

  Using cached matplotlib-3.9.4-cp39-cp39-macosx_11_0_arm64.whl.metadata (11 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached importlib_resources-6.5.2-py3-none-any.whl.metadata (3.9 kB)
Using cached matplotlib-3.9.4-cp39-cp39-macosx_11_0_arm64.whl (7.8 MB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 43.3 MB/s  0:00:00
Using cached importlib_resources-6.5.2-py3-none-any.whl (37 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 39.5 MB/s  0:00:00
Using cached pyparsing-3.3.2-py3-none-any.whl (122 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [matplotlib]8 [matplotlib]
Note: you may need to restart the kernel to use updated packages.


In [12]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame(list(db.nbb_financials.find({}, {"_id": 0})))
print(f"{len(df)} lignes (entreprise x année) chargées")
df.head()


0 lignes (entreprise x année) chargées


""


In [13]:
# On calcule un "chiffre d'affaires ou proxy" pour la suite : le CA réel s'il
# est publié, sinon la marge brute d'exploitation -- décision faite ICI, à
# l'analyse, plutôt que dans le document stocké, pour ne jamais perdre la
# distinction "publié / non publié" dans nbb_financials lui-même.
df["revenu_ou_proxy"] = df["chiffre_affaires"].combine_first(df["marge_brute_exploitation"])
df["solvabilite"] = df["capitaux_propres"] / df["total_actif"]
df["cout_personnel_relatif"] = df["couts_personnel"] / df["revenu_ou_proxy"]

# --- Analyse 1 : santé financière générale, par année ---
by_year = df.groupby("year").agg(
    solvabilite_moyenne=("solvabilite", "mean"),
    tresorerie_mediane=("tresorerie", "median"),
    resultat_net_median=("resultat_net", "median"),
    dettes_financieres_mediane=("dettes_financieres", "median"),
    n_entreprises=("enterprise_number", "nunique"),
).reset_index()
print(by_year)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(by_year["year"], by_year["solvabilite_moyenne"], marker="o")
axes[0].set_title("Solvabilité moyenne (capitaux propres / total actif)")
axes[1].plot(by_year["year"], by_year["resultat_net_median"], marker="o", color="green")
axes[1].set_title("Résultat net médian")
axes[2].plot(by_year["year"], by_year["dettes_financieres_mediane"], marker="o", color="orange")
axes[2].set_title("Dettes financières médianes (contexte hausse des taux 2022-2024)")
plt.tight_layout()
plt.show()

# Conclusion (à rédiger à partir de VOS résultats réels, une phrase suffit) :
# print("Conclusion : ...")


KeyError: 'chiffre_affaires'

In [ ]:
# --- Analyse 2 : comparaison par secteur ---
by_sector = df.explode("sectors").dropna(subset=["sectors"])

sector_stats = by_sector.groupby("sectors").agg(
    resultat_exploitation_moyen=("resultat_exploitation", "mean"),
    cout_personnel_relatif_moyen=("cout_personnel_relatif", "mean"),
    solvabilite_moyenne=("solvabilite", "mean"),
    n_observations=("enterprise_number", "count"),
)

# On ne regarde que les secteurs avec assez d'observations pour être lisibles
# (les indicateurs financiers ne sont pas toujours tous publiés -> beaucoup de NaN)
sector_stats = sector_stats[sector_stats["n_observations"] >= 5].sort_values(
    "resultat_exploitation_moyen", ascending=False
)
print(sector_stats)

sector_stats["resultat_exploitation_moyen"].plot(kind="barh", figsize=(8, max(4, 0.3 * len(sector_stats))))
plt.title("Résultat d'exploitation moyen par secteur (NACE, activité principale)")
plt.tight_layout()
plt.show()

# Conclusion (à rédiger à partir de VOS résultats réels, une phrase suffit) :
# print("Conclusion : ...")


In [ ]:
# --- Analyse 3 : comparaison par région ---
by_region = df.dropna(subset=["region"])

region_stats = by_region.groupby("region").agg(
    resultat_exploitation_moyen=("resultat_exploitation", "mean"),
    cout_personnel_relatif_moyen=("cout_personnel_relatif", "mean"),
    solvabilite_moyenne=("solvabilite", "mean"),
    n_observations=("enterprise_number", "nunique"),
)
print(region_stats)

region_stats[["resultat_exploitation_moyen", "solvabilite_moyenne"]].plot(kind="bar", figsize=(8, 5), secondary_y="solvabilite_moyenne")
plt.title("Résultat d'exploitation moyen et solvabilité, par région")
plt.tight_layout()
plt.show()

# Conclusion (à rédiger à partir de VOS résultats réels, une phrase suffit) :
# print("Conclusion : ...")
